# qlib Native Research Notebook

这份 notebook 现在只负责两件事：
- 配置并调用统一的 `qlib_native` 研究 runner。
- 读取统一产物，展示研究价值、执行质量、风险和稳定性诊断。

训练、打分、原生回测、shared-signal validation、diagnostics 落盘，都由脚本化模块负责，不再在 notebook 内重复维护一条平行逻辑。

## 1. 目标与使用方式

建议的使用姿势：
- 日常研究时，把 `RUN_WORKFLOW = False`，直接复盘已有产物。
- 需要刷新 baseline / candidate 时，把 `RUN_WORKFLOW = True`，让 notebook 直接调用统一 runner。
- 所有主流程参数都集中在一个 `CONFIG` 字典里，避免 notebook 和脚本参数漂移。

## 2. 环境检查与工作流入口

In [9]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qlib_research.core.notebook_workflow import (
    load_native_workflow_artifacts,
    run_native_notebook_workflow,
)


def _safe_last_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.iloc[-1])


def _safe_max_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.max())


def _show_line(frame: pd.DataFrame, x: str, y, title: str, height: int = 360):
    if frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    fig = px.line(frame, x=x, y=y, title=title)
    fig.update_layout(template="plotly_white", height=height)
    fig.show()


def _prepare_heatmap_frame(heatmap_frame: pd.DataFrame) -> pd.DataFrame:
    if heatmap_frame.empty:
        return heatmap_frame
    frame = heatmap_frame.copy()
    if "year" in frame.columns:
        frame["year"] = pd.to_numeric(frame["year"], errors="coerce")
        frame = frame.loc[frame["year"].notna()].copy()
        frame["year"] = frame["year"].astype(int).astype(str)
        frame = frame.set_index("year")
    if "Unnamed: 0" in frame.columns:
        frame = frame.set_index("Unnamed: 0")
    frame = frame.apply(pd.to_numeric, errors="coerce")
    return frame


def _render_return_heatmap(heatmap_frame: pd.DataFrame, title: str, x_label: str, y_label: str) -> None:
    normalized_frame = _prepare_heatmap_frame(heatmap_frame)
    if normalized_frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    numeric_frame = normalized_frame * 100.0
    finite_values = numeric_frame.to_numpy().ravel()
    finite_values = finite_values[pd.notna(finite_values)]
    if len(finite_values):
        abs_values = pd.Series(finite_values).abs()
        robust_bound = float(abs_values.quantile(0.90)) if not abs_values.empty else 1.0
        robust_bound = max(robust_bound, 1.0)
        symmetric_bound = min(robust_bound, 30.0)
    else:
        symmetric_bound = 1.0
    fig = px.imshow(
        numeric_frame,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale=[
            [0.0, "#b2182b"],
            [0.5, "#f7f7f7"],
            [1.0, "#1a9850"],
        ],
        zmin=-symmetric_bound,
        zmax=symmetric_bound,
        origin="lower",
        title=title,
        labels={"x": x_label, "y": y_label, "color": "Net Return %"},
    )
    fig.update_layout(height=420)
    fig.show()


runtime_check = pd.DataFrame(
    [
        {"check": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
        {"check": "Notebook role", "value": "thin native workflow viewer"},
        {"check": "Runner helper", "value": "run_native_notebook_workflow"},
        {"check": "Artifact loader", "value": "load_native_workflow_artifacts"},
    ]
)
display(runtime_check)


,check,value
0,PROJECT_ROOT,/Volumes/Repository/Projects/TradingNexus/Qlib...
1,Notebook role,thin native workflow viewer
2,Runner helper,run_native_notebook_workflow
3,Artifact loader,load_native_workflow_artifacts


## 3. 参数区

In [10]:
RUN_WORKFLOW = False
RECIPE_NAMES = ["baseline", "mae_4w", "binary_4w", "rank_blended", "huber_8w"]
FOCUS_RECIPE = "rank_blended"

CONFIG = {
    "universe_profile": "csi300",
    "panel_path": "artifacts/panels/csi300_weekly_20260410.parquet",
    "execution_panel_path": None,
    "output_dir": "artifacts/native_workflow/csi300_4w_allfeatures_20260410_fe",
    "run_export": "auto_if_missing",
    "start_date": "2016-01-01",
    "end_date": None,
    "batch_size": 200,
    "topk": 10,
    "train_weeks": 260,
    "valid_weeks": 52,
    "eval_count": 52,
    "step_weeks": 4,
    "walk_forward_enabled": True,
    "walk_forward_eval_count": 0,
    "walk_forward_train_weeks": 260,
    "walk_forward_valid_weeks": 52,
    "walk_forward_step_weeks": 4,
    "benchmark_mode": "auto",
    "signal_objective": "huber_regression",
    "label_recipe": "blended_excess_4w_8w",
    "rebalance_interval_weeks": 4,
    "validation-execution-lag-steps": 1,
    "hold_buffer_rank": 15,
    "universe_exit_policy": "retain_quotes_for_existing_positions",
    "min_liquidity_filter": 0.0,
    "min_score_spread": 0.0,
    "industry_max_weight": 0.30,
    "diagnostics_enabled": True,
    "run_validation_comparison": True,
    "publish_model": False,
}

display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))
display(Markdown("`benchmark_mode` 默认按 `universe_profile` 自动映射；也可以手工设为 `000001.SH` 或 `000001.SH@上证指数`。"))
display(Markdown("`walk_forward_eval_count=0` 表示使用全部可评估周；大于 0 时只保留最近 N 个评估周。"))


,value
universe_profile,csi300
panel_path,artifacts/panels/csi300_weekly_20260410.parquet
execution_panel_path,None
output_dir,artifacts/native_workflow/csi300_4w_allfeature...
run_export,auto_if_missing
start_date,2016-01-01
end_date,None
batch_size,200
topk,10
train_weeks,260


`benchmark_mode` 默认按 `universe_profile` 自动映射；也可以手工设为 `000001.SH` 或 `000001.SH@上证指数`。

`walk_forward_eval_count=0` 表示使用全部可评估周；大于 0 时只保留最近 N 个评估周。

## 4. 运行或加载统一研究工作流

In [11]:
workflow_run = run_native_notebook_workflow(
    config_overrides=CONFIG,
    recipe_names=RECIPE_NAMES,
    run_workflow=RUN_WORKFLOW,
)

artifact_view = load_native_workflow_artifacts(
    workflow_run["output_dir"],
    recipe_names=RECIPE_NAMES,
)
workflow_summary = workflow_run["workflow_summary"]

display(Markdown(f"**输出目录**: `{workflow_run['output_dir']}`"))
display(pd.DataFrame([workflow_summary.get("config", CONFIG)]).T.rename(columns={0: "value"}))

if artifact_view["promotion_gate"].empty:
    display(Markdown("当前没有可展示的 promotion gate 输出；如果只运行了 `baseline`，或还没有刷新 workflow，这是正常的。"))
else:
    display(artifact_view["promotion_gate"])


**输出目录**: `/Volumes/Repository/Projects/TradingNexus/QlibResearch/artifacts/native_workflow/csi300_4w_allfeatures_20260410_fe`

,value
universe_profile,csi300
panel_path,/Volumes/Repository/Projects/TradingNexus/Qlib...
execution_panel_path,/Volumes/Repository/Projects/TradingNexus/Qlib...
output_dir,/Volumes/Repository/Projects/TradingNexus/Qlib...
start_date,2016-01-01
end_date,None
batch_size,200
run_export,never
topk,10
train_weeks,260


,recipe,promotion_gate_passed,baseline_topk_unique_score_ratio,candidate_topk_unique_score_ratio,baseline_signal_turnover_proxy,candidate_walk_forward_net_total_return,baseline_walk_forward_net_total_return,candidate_walk_forward_drawdown,baseline_walk_forward_drawdown
0,mae_4w,False,0.963077,0.869231,0.296825,0.110999,0.353547,-0.269641,-0.327252
1,binary_4w,False,0.963077,0.206154,0.296825,0.524436,0.353547,-0.330388,-0.327252
2,rank_blended,True,0.963077,0.898438,0.296825,0.911332,0.353547,-0.225629,-0.327252
3,huber_8w,False,0.963077,0.935938,0.296825,-0.078803,0.353547,-0.285739,-0.327252


## 5. 工作流总览

In [12]:
recipe_overview = artifact_view["recipe_overview"].copy()
if recipe_overview.empty:
    display(Markdown("当前输出目录下还没有 recipe 产物。把 `RUN_WORKFLOW` 改成 `True` 后重新运行上一格。"))
else:
    recipe_overview = recipe_overview.sort_values("recipe").reset_index(drop=True)
    walk_forward_limit = workflow_summary.get("config", {}).get("walk_forward_eval_count")
    if pd.notna(walk_forward_limit) and int(walk_forward_limit) > 0:
        display(Markdown(f"当前 `walk_forward_eval_count={int(walk_forward_limit)}`，只会保留最近 {int(walk_forward_limit)} 个可评估周，而不是全历史 walk-forward。"))
    key_metric_columns = [
        "recipe",
        "rolling_rank_ic_ir",
        "rolling_topk_mean_excess_return_4w",
        "rolling_eval_date_count",
        "rolling_eval_date_start",
        "rolling_eval_date_end",
        "rolling_net_total_return",
        "rolling_max_drawdown",
        "walk_forward_rank_ic_ir",
        "walk_forward_topk_mean_excess_return_4w",
        "walk_forward_eval_date_count",
        "walk_forward_eval_date_start",
        "walk_forward_eval_date_end",
        "walk_forward_net_total_return",
        "walk_forward_max_drawdown",
        "latest_score_dispersion",
        "latest_top10_unique_score_count",
        "latest_actual_hold_count",
        "latest_blocked_sell_count",
    ]
    available_columns = [column for column in key_metric_columns if column in recipe_overview.columns]
    key_metric_frame = recipe_overview[available_columns].copy()
    display(key_metric_frame)
    display(recipe_overview)
    display(Markdown("上表是集中指标看板；`recipe_overview` 则保留完整细项。`requested_feature_count` 对应配置请求特征数，`used_feature_count` 对应实际进入训练的特征数。"))


,recipe,rolling_rank_ic_ir,rolling_topk_mean_excess_return_4w,rolling_eval_date_count,rolling_eval_date_start,rolling_eval_date_end,rolling_net_total_return,rolling_max_drawdown,walk_forward_rank_ic_ir,walk_forward_topk_mean_excess_return_4w,walk_forward_eval_date_count,walk_forward_eval_date_start,walk_forward_eval_date_end,walk_forward_net_total_return,walk_forward_max_drawdown,latest_score_dispersion,latest_top10_unique_score_count,latest_actual_hold_count,latest_blocked_sell_count
0,baseline,-0.129781,-0.009282,13,2025-04-04,2026-03-13,0.088413,-0.042933,0.273351,-0.003405,52,2022-03-11,2026-03-13,0.353547,-0.327252,0.098718,None,12.0,0.0
1,binary_4w,0.240032,-0.004465,13,2025-04-04,2026-03-13,0.350218,-0.055252,-0.241922,-0.007325,52,2022-03-11,2026-03-13,0.524436,-0.330388,0.004965,None,16.0,0.0
2,huber_8w,0.195593,-0.009355,13,2025-03-07,2026-02-06,0.177393,-0.031602,0.274498,-0.007866,51,2022-03-11,2026-02-06,-0.078803,-0.285739,0.104022,None,11.0,0.0
3,mae_4w,-0.098147,-0.006140,13,2025-04-04,2026-03-13,0.170753,-0.038270,0.264113,-0.005632,52,2022-03-11,2026-03-13,0.110999,-0.269641,0.035174,None,12.0,0.0
4,rank_blended,-0.177664,-0.011863,13,2025-03-07,2026-02-06,0.450606,-0.045351,0.154840,-0.001752,51,2022-03-11,2026-02-06,0.911332,-0.225629,0.287801,None,14.0,0.0


,recipe,path,requested_feature_count,used_feature_count,rolling_rank_ic_ir,rolling_topk_mean_excess_return_4w,rolling_eval_date_count,rolling_eval_date_start,rolling_eval_date_end,rolling_report_date_start,...,walk_forward_excess_total_return,walk_forward_max_drawdown,walk_forward_excess_drawdown,walk_forward_cost_drag,walk_forward_turnover_mean,latest_score_dispersion,latest_top10_unique_score_count,latest_actual_hold_count,latest_blocked_sell_count,latest_score_rows
0,baseline,/Volumes/Repository/Projects/TradingNexus/Qlib...,None,79,-0.129781,-0.009282,13,2025-04-04,2026-03-13,2025-05-02,...,0.269345,-0.327252,-0.109408,0.007134,0.265409,0.098718,None,12.0,0.0,300
1,binary_4w,/Volumes/Repository/Projects/TradingNexus/Qlib...,None,79,0.240032,-0.004465,13,2025-04-04,2026-03-13,2025-05-02,...,0.440234,-0.330388,-0.190302,0.006708,0.249151,0.004965,None,16.0,0.0,300
2,huber_8w,/Volumes/Repository/Projects/TradingNexus/Qlib...,None,79,0.195593,-0.009355,13,2025-03-07,2026-02-06,2025-04-04,...,-0.157074,-0.285739,-0.288383,0.007700,0.282546,0.104022,None,11.0,0.0,300
3,mae_4w,/Volumes/Repository/Projects/TradingNexus/Qlib...,None,79,-0.098147,-0.006140,13,2025-04-04,2026-03-13,2025-05-02,...,0.026796,-0.269641,-0.187664,0.008600,0.318904,0.035174,None,12.0,0.0,300
4,rank_blended,/Volumes/Repository/Projects/TradingNexus/Qlib...,None,79,-0.177664,-0.011863,13,2025-03-07,2026-02-06,2025-04-04,...,0.833061,-0.225629,-0.112243,0.007616,0.278762,0.287801,None,14.0,0.0,300


上表是集中指标看板；`recipe_overview` 则保留完整细项。`requested_feature_count` 对应配置请求特征数，`used_feature_count` 对应实际进入训练的特征数。

## 6. 研究价值看板

In [13]:
available_recipes = artifact_view["recipe_names"]
focus_recipe = FOCUS_RECIPE if FOCUS_RECIPE in available_recipes else (available_recipes[0] if available_recipes else None)

if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    feature_prefilter = recipe_data["feature_prefilter"]
    signal_diagnostics = recipe_data["signal_diagnostics"].copy()
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()

    research_value_board = pd.DataFrame(
        [
            {
                "recipe": focus_recipe,
                "requested_feature_count": _safe_max_numeric(feature_prefilter, "requested_feature_count"),
                "used_feature_count": _safe_max_numeric(feature_prefilter, "selected_feature_count"),
                "latest_score_dispersion": _safe_last_numeric(signal_diagnostics, "score_dispersion"),
                "latest_top10_unique_score_count": _safe_last_numeric(signal_diagnostics, "topk_unique_score_count"),
                "latest_top10_unique_score_ratio": _safe_last_numeric(signal_diagnostics, "topk_unique_score_ratio"),
                "latest_actual_hold_count": _safe_last_numeric(portfolio_diagnostics, "actual_hold_count"),
                "latest_blocked_sell_count": _safe_last_numeric(portfolio_diagnostics, "blocked_sell_count"),
                "slice_row_count": int(len(slice_regime_summary)),
            }
        ]
    )
    display(research_value_board)

    if not signal_diagnostics.empty and "feature_date" in signal_diagnostics.columns:
        signal_diagnostics["feature_date"] = pd.to_datetime(signal_diagnostics["feature_date"])
        _show_line(
            signal_diagnostics,
            x="feature_date",
            y=["score_dispersion", "topk_unique_score_ratio"],
            title=f"{focus_recipe}: 分数离散度与 TopK 唯一分数比例",
        )


,recipe,requested_feature_count,used_feature_count,latest_score_dispersion,latest_top10_unique_score_count,latest_top10_unique_score_ratio,latest_actual_hold_count,latest_blocked_sell_count,slice_row_count
0,rank_blended,None,None,0.287801,None,1.0,14.0,0.0,69


## 7. 执行与风险诊断

In [14]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    execution_diff_summary = recipe_data["execution_diff_summary"].copy()
    rolling_native_report = recipe_data["rolling_native_report"].copy()
    walk_forward_native_report = recipe_data["walk_forward_native_report"].copy()
    rolling_monthly_heatmap = recipe_data["rolling_native_monthly_return_heatmap"].copy()
    rolling_annual_heatmap = recipe_data["rolling_native_annual_return_heatmap"].copy()
    walk_forward_monthly_heatmap = recipe_data["walk_forward_native_monthly_return_heatmap"].copy()
    walk_forward_annual_heatmap = recipe_data["walk_forward_native_annual_return_heatmap"].copy()

    display(portfolio_diagnostics.tail(12) if not portfolio_diagnostics.empty else pd.DataFrame())
    display(execution_diff_summary if not execution_diff_summary.empty else pd.DataFrame())

    if not portfolio_diagnostics.empty and "trade_date" in portfolio_diagnostics.columns:
        portfolio_diagnostics["trade_date"] = pd.to_datetime(portfolio_diagnostics["trade_date"])
        _show_line(
            portfolio_diagnostics,
            x="trade_date",
            y=["target_hold_count", "actual_hold_count", "blocked_sell_count"],
            title=f"{focus_recipe}: 目标持仓、实际持仓与 blocked sell",
        )

    if not rolling_native_report.empty and "datetime" in rolling_native_report.columns:
        rolling_native_report["datetime"] = pd.to_datetime(rolling_native_report["datetime"])
        _show_line(
            rolling_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: rolling 净值 vs benchmark",
            height=420,
        )

    if not walk_forward_native_report.empty and "datetime" in walk_forward_native_report.columns:
        walk_forward_native_report["datetime"] = pd.to_datetime(walk_forward_native_report["datetime"])
        _show_line(
            walk_forward_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: walk-forward 净值 vs benchmark",
            height=420,
        )

    _render_return_heatmap(
        rolling_monthly_heatmap,
        title=f"{focus_recipe}: rolling 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        rolling_annual_heatmap,
        title=f"{focus_recipe}: rolling 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )
    _render_return_heatmap(
        walk_forward_monthly_heatmap,
        title=f"{focus_recipe}: walk-forward 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        walk_forward_annual_heatmap,
        title=f"{focus_recipe}: walk-forward 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )


,signal_date,trade_date,target_hold_count,actual_hold_count,blocked_sell_count,blocked_sell_codes,residual_hold_count,rebalance_interval_steps,hold_buffer_rank,coverage,score_dispersion,score_unique_count,topk_unique_score_ratio,topk_overlap_prev,future_return_top_bottom_decile_spread,excess_return_top_bottom_decile_spread,recipe,bundle
50,2025-03-07,2025-04-04,10,17,0,NaN,14,4,10,297.0,0.295618,290.0,1.0,0.0,-0.028074,-0.028074,rank_blended,walk_forward
51,2025-04-04,2025-05-02,10,17,0,NaN,15,4,10,299.0,0.160791,289.0,1.0,0.1,0.030501,0.030501,rank_blended,walk_forward
52,2025-05-02,2025-05-30,10,12,0,NaN,3,4,10,299.0,0.359582,296.0,1.0,0.0,-0.000954,-0.000954,rank_blended,walk_forward
53,2025-05-30,2025-06-27,10,12,0,NaN,10,4,10,297.0,0.325294,260.0,1.0,0.2,-0.026002,-0.026002,rank_blended,walk_forward
54,2025-06-27,2025-07-25,10,12,0,NaN,9,4,10,299.0,0.422906,299.0,1.0,0.3,-0.063417,-0.063417,rank_blended,walk_forward
55,2025-07-25,2025-08-22,10,12,0,NaN,12,4,10,299.0,0.032498,108.0,0.7,0.1,-0.029546,-0.029546,rank_blended,walk_forward
56,2025-08-22,2025-09-19,10,17,0,NaN,8,4,10,299.0,0.471095,299.0,1.0,0.2,-0.124709,-0.124709,rank_blended,walk_forward
57,2025-09-19,2025-10-17,10,17,0,NaN,17,4,10,299.0,0.163799,296.0,1.0,0.0,0.017275,0.017275,rank_blended,walk_forward
58,2025-10-17,2025-11-14,10,17,0,NaN,16,4,10,300.0,0.217877,288.0,1.0,0.1,0.018183,0.018183,rank_blended,walk_forward
59,2025-11-14,2025-12-12,10,17,0,NaN,17,4,10,300.0,0.605101,300.0,1.0,0.2,-0.000648,-0.000648,rank_blended,walk_forward


,recipe,bundle,native_final_net_value,validation_final_net_value,native_minus_validation_return,native_max_drawdown,validation_max_drawdown
0,rank_blended,rolling,1.450606e+06,1.258595e+06,0.192011,-0.045351,-0.045298
1,rank_blended,walk_forward,1.911332e+06,1.356241e+06,0.555091,-0.225629,-0.276291


## 8. 切片稳定性与特征重要性

In [15]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()
    rolling_feature_importance = recipe_data["rolling_feature_importance"].copy()

    if not slice_regime_summary.empty:
        display(slice_regime_summary.sort_values(["bundle", "slice_type", "coverage"], ascending=[True, True, False]).head(30))
    else:
        display(pd.DataFrame())

    if not rolling_feature_importance.empty:
        top_features = (
            rolling_feature_importance.groupby("feature", as_index=False)["importance_gain"]
            .mean()
            .sort_values("importance_gain", ascending=False)
            .head(20)
        )
        display(top_features)
        fig = px.bar(
            top_features.sort_values("importance_gain", ascending=True),
            x="importance_gain",
            y="feature",
            orientation="h",
            title=f"{focus_recipe}: rolling 平均特征重要性（gain）",
        )
        fig.update_layout(template="plotly_white", height=520)
        fig.show()


,recipe,bundle,slice_type,slice_value,coverage,score_dispersion,mean_future_return_4w,mean_excess_return_4w
0,rank_blended,rolling,feature_year,2025,3286,0.394267,0.019156,-0.000957
1,rank_blended,rolling,feature_year,2026,599,0.241182,0.002089,-0.004163
24,rank_blended,rolling,l1_name,电子,403,0.415679,0.027042,0.009147
33,rank_blended,rolling,l1_name,非银金融,338,0.356761,0.005435,-0.012162
32,rank_blended,rolling,l1_name,银行,302,0.328577,0.007056,-0.011050
23,rank_blended,rolling,l1_name,电力设备,293,0.390262,0.038277,0.020151
35,rank_blended,rolling,l1_name,NaN,271,0.319863,0.014050,-0.004025
11,rank_blended,rolling,l1_name,医药生物,265,0.392455,-0.003286,-0.021207
7,rank_blended,rolling,l1_name,交通运输,197,0.365218,0.012416,-0.005548
28,rank_blended,rolling,l1_name,计算机,176,0.324897,-0.007069,-0.025087


,feature,importance_gain
18,dea,355.174946
46,macro_recovery_x_mom_8w,340.250993
13,cfo_to_ni_ttm,275.177961
20,dif,250.268558
43,macd_hist,222.435782
69,rev_8w,199.423070
21,downside_volatility_8w,190.720044
50,mom_8w,183.520758
35,gpm_ttm_12q_delta,172.717571
53,ni_ttm,163.148770


## 9. 最新一期信号快照

In [16]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    latest_score_frame = artifact_view["recipes"][focus_recipe]["latest_score_frame"].copy()
    if latest_score_frame.empty:
        display(Markdown("当前 recipe 没有最新一期快照。"))
    else:
        display(latest_score_frame.head(20))
        if {"l1_name", "score"}.issubset(latest_score_frame.columns):
            industry_snapshot = (
                latest_score_frame.head(50)
                .groupby("l1_name", dropna=False)
                .agg(stock_count=("instrument", "count"), mean_score=("score", "mean"))
                .reset_index()
                .sort_values(["mean_score", "stock_count"], ascending=[False, False])
            )
            display(industry_snapshot)

display(Markdown("如果要发布线上 snapshot，请把 `CONFIG['publish_model']` 改成 `True`，然后重新运行 workflow cell。"))


,datetime,instrument,score,open,close,volume,amount,future_return_4w,future_return_8w,label_excess_return_4w,label_excess_return_8w,model_label_raw,model_label,in_csi300,l1_name,l2_name,l3_name,macro_phase,macro_industry_match,feature_date
0,2026-04-10,601009.SH,0.551670,11.17,11.02,1255119,1392973.435,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-10
1,2026-04-10,600919.SH,0.535162,10.81,10.81,2036486,2199685.233,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-10
2,2026-04-10,601229.SH,0.358763,9.85,9.62,1137305,1100957.356,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-10
3,2026-04-10,002142.SZ,0.348335,30.01,30.10,511346,1548367.820,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-10
4,2026-04-10,601169.SH,0.345369,5.43,5.40,2589861,1402878.186,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,城商行Ⅱ,城商行Ⅲ,REFLATION,0,2026-04-10
5,2026-04-10,688223.SH,0.338235,6.25,6.44,4366883,2808521.990,NaN,NaN,NaN,NaN,NaN,NaN,True,电力设备,光伏设备,光伏电池组件,REFLATION,1,2026-04-10
6,2026-04-10,300316.SZ,0.316245,41.21,41.39,808015,3363841.177,NaN,NaN,NaN,NaN,NaN,NaN,True,电力设备,光伏设备,光伏加工设备,REFLATION,1,2026-04-10
7,2026-04-10,688126.SH,0.281369,16.76,17.51,721909,1258964.355,NaN,NaN,NaN,NaN,NaN,NaN,True,电子,半导体,半导体材料,REFLATION,0,2026-04-10
8,2026-04-10,600377.SH,0.278746,11.74,11.53,264715,305640.653,NaN,NaN,NaN,NaN,NaN,NaN,True,交通运输,铁路公路,高速公路,REFLATION,0,2026-04-10
9,2026-04-10,601077.SH,0.271881,7.04,7.18,1524213,1079916.272,NaN,NaN,NaN,NaN,NaN,NaN,True,银行,农商行Ⅱ,农商行Ⅲ,REFLATION,0,2026-04-10


,l1_name,stock_count,mean_score
11,电力设备,3,0.235668
14,美容护理,1,0.222161
16,银行,15,0.214910
12,电子,2,0.199115
18,食品饮料,3,0.182367
8,有色金属,2,0.174588
15,计算机,2,0.109075
2,公用事业,1,0.104674
0,交通运输,4,0.092736
17,非银金融,4,0.085613


如果要发布线上 snapshot，请把 `CONFIG['publish_model']` 改成 `True`，然后重新运行 workflow cell。

## 10. 后续动作

建议优先按这个顺序继续推进：
- 先比较 `baseline` 与 candidate 的 `execution_diff_summary`，确认收益问题到底来自信号还是执行。
- 再看 `research_value_board` 与 `slice_regime_summary`，确认分数粒度、年份稳定性、行业集中度是否改善。
- 最后才决定是否发布 snapshot，避免把“会排不会赚”的配方直接推向下游。